# 03. Postprocessing (steps ⑥-⑭)

| | |
|---|---|
| Input | `3D_raw_online.xlsx` from 02, calibration NPZ |
| Output | `3D_final.xlsx` (Result / Legend / Plot_Aux / Summary / Configuration / Debug) |
| GPU | Not required |

One cell per step, run in order. Intermediate results are in `ctx.df`.
The last cell runs all steps at once and gives the same result.

In [ ]:
#@title 0. Setup (clone repository, install packages, mount Drive)
REPO_URL = "https://github.com/dodokim777-boop/underwater-gripper-tracking.git"  #@param {type:"string"}

import os, sys, subprocess, warnings
REPO_DIR = '/content/' + REPO_URL.rstrip('/').split('/')[-1].removesuffix('.git')
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '-q'], check=False)
%pip install -q -r {REPO_DIR}/requirements.txt

from google.colab import drive
drive.mount('/content/drive')

sys.path.insert(0, REPO_DIR)
import pandas as pd
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)
print('Ready:', REPO_DIR)

In [ ]:
#@title 1. Settings file
SETTINGS_PATH = "/content/drive/MyDrive/files/experiment_settings.yaml"  #@param {type:"string"}

import shutil
from main_code import settings

if not os.path.exists(SETTINGS_PATH):
    os.makedirs(os.path.dirname(SETTINGS_PATH), exist_ok=True)
    shutil.copy(os.path.join(REPO_DIR, 'experiment_settings.yaml'), SETTINGS_PATH)
    raise RuntimeError(f'Template copied to {SETTINGS_PATH}. Edit the required values and run this cell again.')

settings.configure(SETTINGS_PATH)
settings.print_summary('postprocessing')
settings.check('postprocessing')

In [ ]:
#@title 2. Load raw Excel and calibration
from main_code.postprocessing import run_postprocessing as P
ctx = P.step_load()
print('Frames:', len(ctx.df))

In [ ]:
#@title ⑥ Outlier removal (Hampel)
ctx = P.step06_remove_outliers(ctx)
print('Outliers:', ctx.n_outliers)

In [ ]:
#@title ⑦ Anchor snapshot
ctx = P.step07_freeze_anchors(ctx)
print('Anchor frames:', int(ctx.df['Anchor_Raw_Valid'].sum()))

In [ ]:
#@title ⑧ Process noise σa
ctx = P.step08_estimate_noise(ctx)

In [ ]:
#@title ⑨ Recovery mode
ctx = P.step09_classify_recovery(ctx)
print(ctx.df['Recovery_Mode'].value_counts().to_string())

In [ ]:
#@title ⑩ Single-view recovery
ctx = P.step10_recover_missing(ctx)
print(ctx.df['Filled_Type'].value_counts().to_string())
print(ctx.df['Hidden_Stop_Reason'].value_counts().to_string())

In [ ]:
#@title ⑪ Work zone · GRAB
ctx = P.step11_zone_and_grab(ctx)
print('GRAB events:', ctx.n_grab)
print(ctx.df['Coord_State'].value_counts().to_string())

In [ ]:
#@title ⑫ Display-only zone bridging
ctx = P.step12_display_zone(ctx)

In [ ]:
#@title ⑬ Display-only RTS smoothing
ctx = P.step13_smooth_rts(ctx)

In [ ]:
#@title ⑭ Velocity
ctx = P.step14_velocity(ctx)
print('Velocity frames:', int(ctx.df['Vx_cm_s'].notna().sum()))

In [ ]:
#@title Save 3D_final.xlsx
ctx = P.step_save(ctx)

In [ ]:
#@title Result plot
import matplotlib.pyplot as plt
res = ctx.sheets['Result']
fig, axes = plt.subplots(4, 1, figsize=(14, 9), sharex=True)
for ax, c in zip(axes, ['X_cm', 'Y_cm', 'Z_cm']):
    for state, color in (('DETECTED', 'k'), ('RECOVERED', 'tab:orange')):
        m = res['State'].eq(state)
        ax.plot(res.loc[m, 'Time_s'], res.loc[m, c], '.', ms=2, color=color, label=state)
    ax.set_ylabel(c); ax.grid(alpha=.3)
axes[0].legend(markerscale=5)
axes[3].plot(res['Time_s'], res['Speed_3D_cm_s'], lw=.8); axes[3].set_ylabel('Speed_3D [cm/s]'); axes[3].grid(alpha=.3)
axes[-1].set_xlabel('Time [s]'); plt.tight_layout(); plt.show()

In [ ]:
#@title Run all steps at once (same result)
from main_code.postprocessing.run_postprocessing import run_postprocessing
df_final = run_postprocessing()